# Reproduction: LLM2Vec on the course testbed

Pair 02, Baptiste Pras and Raphael Leonardi.

This notebook does not train or encode anything, it runs in a few seconds. It reads the small files in `results/`:
the testbed scores (`results/*.json`), the scores of each query (`per_query.json`) and the compute time (`compute.json`).
`python analysis.py` makes these files again from our runs and logs.

The key result we reproduce is **Table 11 of the paper**: the authors' adapters for S-LLaMA-1.3B (MNTP then unsupervised SimCSE), but evaluated with our code.
Then we do a scaled down **Table 5** (the ablation of the three steps) on Qwen2.5-0.5B, with our own training.

In [ ]:
import numpy as np
import pandas as pd

import analysis
from analysis import testbed

P = analysis.per_query()
pd.set_option("display.precision", 3)


def ndcg(system, name):
    return np.array(P[name][system]["ndcg10"])

## 1. The authors' adapters with our code (Table 11)

We load `princeton-nlp/Sheared-LLaMA-1.3B`, merge `McGill-NLP/LLM2Vec-Sheared-LLaMA-mntp`, then merge `McGill-NLP/LLM2Vec-Sheared-LLaMA-mntp-unsup-simcse`.
Then we encode with our `encoder.py`: bidirectional mask, the instruction of Table 10 on the queries only, mean pooling without the instruction and without the BOS token.
Two things are not the same as in the paper: the testbed cuts each document after 200 words (MTEB keeps all of it), and we score with `pytrec_eval` from the testbed, not with MTEB.

In [ ]:
paper = {"scifact": 38.31, "nfcorpus": 15.73, "arguana": 43.64}  # Table 11, column S-LLaMA-1.3B
rows = []
for name in testbed.DATASETS:
    ours = 100 * ndcg("sllama", name).mean()
    rows.append({"dataset": name, "paper (Table 11)": paper[name], "our code": round(ours, 2),
                 "difference": round(ours - paper[name], 2)})
pd.DataFrame(rows).set_index("dataset")

The difference is less than 0.8 point on the three datasets. So our encoding (mask, instruction, pooling) gives the same thing as the authors' code, and the 200 words cut does not change much.
This is important for the rest: when our own models are worse or better than the paper, it comes from the training, not from a bug in the evaluation.

## 2. Ablation on Qwen2.5-0.5B (scaled down Table 5)

Table 5 gives the average on 15 MTEB tasks (not only retrieval) for S-LLaMA-1.3B with mean pooling. So we can not compare the numbers, only the **order** of the six variants.

Our setting (see `train.py`): Qwen2.5-0.5B, LoRA r=16, 1000 steps, batch 32, max length 128, for MNTP and for SimCSE.
The paper trains on Wikipedia, we train on the documents of the three testbed corpora (`prepare_corpus.py`), never on queries or qrels.

In [ ]:
paper_t5 = {"uni": 33.03, "bi": 30.26, "bi-mntp": 42.10, "uni-simcse": 44.95, "bi-simcse": 44.46, "full": 52.40}
names = {"uni": "Uni", "bi": "Bi", "bi-mntp": "Bi + MNTP", "uni-simcse": "Uni + SimCSE",
         "bi-simcse": "Bi + SimCSE", "full": "Bi + MNTP + SimCSE"}
rows = []
for s, score in paper_t5.items():
    row = {"variant": names[s], "paper, S-LLaMA (Table 5)": score}
    for name in ["scifact", "nfcorpus"]:
        row[name] = round(100 * ndcg(s, name).mean(), 1)
    rows.append(row)
ablation = pd.DataFrame(rows).set_index("variant")
ablation["paper rank"] = ablation["paper, S-LLaMA (Table 5)"].rank(ascending=False).astype(int)
ablation["our rank"] = ablation[["scifact", "nfcorpus"]].mean(axis=1).rank(ascending=False).astype(int)
ablation

Each step alone, with a paired t-test over the queries (`testbed.compare`). We make 14 tests here, so with Bonferroni a p-value must be below 0.05 / 14 = 0.0036.

In [ ]:
steps = [("bi", "uni", "bidirectional attention alone"),
         ("bi-mntp", "bi", "MNTP after Bi"),
         ("bi-simcse", "bi", "SimCSE after Bi"),
         ("bi-simcse", "uni-simcse", "Bi or Uni, both with SimCSE"),
         ("full", "bi-simcse", "MNTP before SimCSE"),
         ("q15-full", "full", "Qwen 1.5B or 0.5B, full recipe"),
         ("full", "sllama", "our Qwen 0.5B or authors' S-LLaMA 1.3B")]
rows = []
for a, b, what in steps:
    for name in ["scifact", "nfcorpus"]:
        c = testbed.compare(ndcg(a, name), ndcg(b, name))
        rows.append({"a - b": f"{a} - {b}", "what": what, "dataset": name,
                     "delta": c["delta"], "ci95": c["ci95"], "p": c["p"]})
tests = pd.DataFrame(rows)
tests["significant (Bonferroni)"] = tests["p"] < 0.05 / len(tests)
tests.style.format({"delta": "{:+.3f}", "p": "{:.2g}"})

What we see, and what is the same as the paper:

* **Bi alone does nothing** (paper: it is even worse than Uni). Our difference is noise on both datasets.
* **SimCSE is the step that matters most** for retrieval: from about 0.2 to 0.58 on SciFact.
* **Bi is better than Uni when there is SimCSE** (+0.09 on SciFact, +0.04 on NFCorpus, both significant). In the paper Bi + SimCSE and Uni + SimCSE are almost the same with S-LLaMA, Bi wins with Mistral.
* **MNTP alone helps a little** but much less than in the paper (S-LLaMA: from 30.3 to 42.1).
* **MNTP before SimCSE does not help us**: Full and Bi + SimCSE are the same (noise). In the paper this is the best variant, +8 points. This is the main thing we do not reproduce. Our guess: 1000 steps of MNTP on 16k documents is too small, the authors' S-LLaMA MNTP adapter helps a lot in our sanity check (hit@10 from 0.28 to 0.72 on a small SciFact subset).
* The bigger model helps a bit: Qwen 1.5B is better than 0.5B on both datasets (p = 0.03 and 0.005), but this is not below the Bonferroni limit, so we say it carefully.
* Our Qwen 0.5B is better than the authors' S-LLaMA 1.3B on SciFact (+0.17), only a little on NFCorpus. But our model is trained on the testbed documents (in domain), S-LLaMA was trained on Wikipedia.

## 3. Same budget for an encoder (the Ettin question)

Same SimCSE training (1000 steps, batch 32, length 128, LoRA, same corpus) on two encoders of about 150M parameters: `jhu-clsp/ettin-encoder-150m` and `answerdotai/ModernBERT-base`.
They are already bidirectional, so there is no MNTP step. The paper uses attention dropout 0.3 for SimCSE on decoders. We first used 0.3, then we tried 0.1 (the value of the original SimCSE on BERT) **after** we saw the first results, so we report both.

In [ ]:
systems = {"full": "LLM2Vec Qwen2.5-0.5B", "ettin": "Ettin-150m, no training", "ettin-simcse": "Ettin + SimCSE, dropout 0.3",
           "ettin-simcse-d01": "Ettin + SimCSE, dropout 0.1", "modernbert": "ModernBERT, no training",
           "modernbert-simcse": "ModernBERT + SimCSE, dropout 0.3", "modernbert-simcse-d01": "ModernBERT + SimCSE, dropout 0.1",
           "bm25": "BM25", "bge": "bge-small-en-v1.5"}
rows = []
for s, label in systems.items():
    row = {"system": label}
    for name in testbed.DATASETS:
        row[name] = ndcg(s, name).mean() if s in P[name] else None
    rows.append(row)
pd.DataFrame(rows).set_index("system")

With the same training the encoders stay far below LLM2Vec. ModernBERT gets a bit better with SimCSE at dropout 0.3, Ettin gets even worse than before training on SciFact.
ArguAna is the exception: there the query is a full argument, so it looks like a document.
Why? The mean cosine on SciFact (`results/encoder_geometry.json`, 150 queries) shows it: for the encoders the documents spread out with SimCSE, but the queries stay close together.
So all the queries find the same few documents. LLM2Vec on Qwen does not have this problem.

In [ ]:
geometry = pd.DataFrame(analysis.geometry()).set_index("run")
geometry["distinct docs in all top 10 (of 3000)"] = [P["scifact"][r]["distinct_top10"] for r in geometry.index]
geometry

SimCSE only sees pieces of documents of 128 tokens during training, and the SciFact queries are short sentences. The decoder seems to transfer to short queries (thanks to its pretraining and to the instruction), the small encoders do not.
So with this budget the answer to the Ettin question is not "encoders win". But this is also not a fair test of the Ettin claim: LLM2Vec is a recipe made for decoders, and we did not try a recipe made for encoders (for example SimCSE on short sentences, as in the original SimCSE). We put this in "what we did not test".

## 4. Our testbed row

`results.json` is LLM2Vec on Qwen2.5-0.5B (full recipe), compared with `bge`, the closest dense baseline. Below the same row against BM25, and the 1.5B model.

In [ ]:
rows = []
for s in ["full", "q15-full"]:
    for b in ["bge", "bm25"]:
        for name in testbed.DATASETS:
            c = testbed.compare(ndcg(s, name), ndcg(b, name))
            rows.append({"ours": s, "baseline": b, "dataset": name, "ours nDCG@10": ndcg(s, name).mean(),
                         "baseline nDCG@10": ndcg(b, name).mean(), "delta": c["delta"], "ci95": c["ci95"], "p": c["p"]})
pd.DataFrame(rows).style.format({"ours nDCG@10": "{:.3f}", "baseline nDCG@10": "{:.3f}", "delta": "{:+.3f}", "p": "{:.2g}"})

We lose clearly against bge and BM25 on all datasets. The only number close to a baseline is ArguAna for the 1.5B model, which is about the same as BM25.
bge-small has 33M parameters but it was trained with a lot of labelled pairs (supervised contrastive learning). Our models only had unsupervised SimCSE.
The paper also shows this: the unsupervised LLM2Vec models are far from the supervised ones (Table 11 against Table 12).

## 5. Compute budget

All on one Mac (Apple M5, 24 GB, PyTorch MPS, bfloat16). One night the Mac went to sleep during the Qwen 1.5B SimCSE training: some steps took more than 1000 s.
We replace these steps by the median step to count only real compute (`analysis.training_hours`). Not counted: tests, debugging, the failed runs before the memory fix, and the authors' training of S-LLaMA.

In [ ]:
compute = pd.DataFrame(analysis.compute())
train = compute[compute["what"] == "train"][["run", "model", "step", "steps", "hours_raw", "hours", "sleep_steps"]]
train = train.astype({"steps": int, "sleep_steps": int})
evaluation = compute[compute["what"] == "eval"][["run", "datasets", "hours"]].astype({"datasets": int})
print(f"training {train['hours'].sum():.1f} h (with sleep {train['hours_raw'].sum():.1f} h), "
      f"evaluation {evaluation['hours'].sum():.1f} h, total {train['hours'].sum() + evaluation['hours'].sum():.1f} h")
train

In [ ]:
evaluation

For one Qwen2.5-0.5B model the full recipe costs about 2 h of MNTP and 3.8 h of SimCSE on the Mac, and encoding the three corpora takes about 35 min.
The paper does each step in 1000 steps on one 80GB A100: MNTP takes 100 minutes and SimCSE 3 hours for the 7B and 8B models (Section 3.1). So our Mac needs about the same time for a 0.5B model as an A100 for a 7B model.

## 6. What did not work

* MNTP before SimCSE gives nothing on our setting (Section 2), when it is the best step in the paper.
* SimCSE on encoders with the same budget breaks them on short queries (Section 3). Changing the dropout from 0.3 to 0.1 made it worse.
* At the start the evaluation crashed on MPS (out of memory): every new batch length makes a new shape and the memory grows. We now pad each batch to a multiple of 64 tokens (`encoder.py`).
* On MPS with bfloat16 the default mask of transformers is not causal, so we always give our own 4D mask (`test_encoder.py` shows it).